In [ ]:
""" 
Esercizio 1
Creare un CSV fittizio di 1 milione di righe con colonne: ID, Prodotto, Prezzo, Quantità, Regione. 
Leggerlo a chunk di 100.000 righe e calcolare la media dei prezzi per chunk. 
Stampare il massimo e il minimo delle medie.
"""

import numpy as np 
import pandas as pd 
from pathlib import Path 

# REPERISCE IL PERCORSO DELLA CARTELLA IN CUI E' PRESENTE LO SCRIPT ATTUALE
try:
    percorso_cartella = Path(__file__).resolve().parent
except:
    percorso_cartella = Path.cwd()

n_transazioni = 1_000_000
rng = np.random.default_rng(seed = 111)

# ELENCO GENERAZIONI DATI CASUALI 
ids = [ f"Tran_{i:010d}" for i in range(1, n_transazioni + 1)]
prodotti = ["Smartphone","Smartwatch","Laptop","Tablet","Pc Desktop","Cuffie bluetooth","Cuffie"]
regioni_italiane = [
    "Abruzzo", "Basilicata", "Calabria", "Campania", "Emilia-Romagna",
    "Friuli-Venezia Giulia", "Lazio", "Liguria", "Lombardia", "Marche",
    "Molise", "Piemonte", "Puglia", "Sardegna", "Sicilia", "Toscana",
    "Trentino-Alto Adige", "Umbria", "Valle d'Aosta", "Veneto"
]
prezzi = rng.uniform(10.0, 1000.0, n_transazioni)
quantita = rng.integers(1, 10, n_transazioni)

# ISTANZA DATAFRAME 
store_df = pd.DataFrame({
    "ID": ids,
    "Prodotto": rng.choice(prodotti, n_transazioni),
    "Regione": rng.choice(regioni_italiane, n_transazioni),
    "Prezzo": prezzi.round(2),
    "Valuta": "EUR",
    "Quantita": quantita
})

print("Prime righe dataframe:\n", store_df.head())

# SERIALIZZAZIONE IN FILE DI TESTO CSV 
store_df.to_csv(percorso_cartella/"store.csv", index=False)

# PRIMA LETTURA A CHUNK (Per Esercizio 1)
chunk_iter = pd.read_csv(percorso_cartella/"store.csv", chunksize=100_000)

medie_per_chunk = []
i = 1
for chunk in chunk_iter:
    medie_per_chunk.append(chunk["Prezzo"].mean())
    print(f"Chunk {i} ELABORATO.")
    i += 1

print("\nCALCOLO MEDIA PREZZO PER CHUNK\n")
for j, media in enumerate(medie_per_chunk):
    # CORRETTO: usando la funzione round() standard
    print(f"Chunk {j+1} media: {round(media, 2)}")

print("\nIL MASSIMO VALORE DELLE VARIE MEDIE:", round(max(medie_per_chunk), 2))
print("IL MINIMO VALORE DELLE VARIE MEDIE:", round(min(medie_per_chunk), 2))


""" 
Esercizio 2
Utilizzando HDF5, salvare lo stesso dataset in più chiavi (es. per regione) 
e leggere solo un subset alla volta, calcolando la somma totale delle vendite per quella regione.
"""

# RICREIAMO L'ITERATORE (perché quello di prima è stato esaurito)
chunk_iter = pd.read_csv(percorso_cartella/"store.csv", chunksize=100_000)

# SERIALIZZAZIONE CHUNK DEL CSV IN FILE HDF5 
for i, chunk in enumerate(chunk_iter):
    for regione, gruppo in chunk.groupby("Regione"):

        # pulisco il nome della chiave altrimenti compare il warning NaturalNameWarning per key hdf5
        chiave = (
            str(regione)
            .strip()
            .replace(" ", "_")
            .replace("-", "_")
            .replace("'", "")
            .lower()
        )
        # Salvo il sottogruppo nella chiave di appartenenza
        gruppo.to_hdf(percorso_cartella/'store_hdf5.hdf5', key=chiave, mode='a', format='table', append=True)

print("\n--- LETTURA DA HDF5 E CALCOLO VENDITE PER REGIONE ---\n")

# APRIAMO IL FILE HDF5 IN SOLA LETTURA 
with pd.HDFStore(percorso_cartella/'store_hdf5.hdf5', mode='r') as store:
    for chiave in store.keys():
        df_regione = store[chiave]

        # CORRETTO: "Prezzo" con la P maiuscola come nel DataFrame originale
        totale_vendite = (df_regione["Quantita"] * df_regione["Prezzo"]).sum()

        print(f"Totale vendite per la regione {chiave.lstrip('/')}: {totale_vendite:,.2f} €")

In [ ]:
""" 
Esercizio 3
Creare un dataset grande, 
filtrare le righe con Prezzo > 50 per ogni chunk e salvare in un file Parquet finale, usando un writer continuo. 
Verificare la correttezza del file finale contando le righe e confrontando con il filtro applicato.
"""

import numpy as np 
import pandas as pd 
from pathlib import Path 
import pyarrow as pa
import pyarrow.parquet as pq


# ==========================================
# 1. IMPOSTAZIONE DEI PERCORSI E DEI DATI
# ==========================================

# Reperisce il percorso della cartella in cui è presente lo script attuale
try:
    percorso_cartella = Path(__file__).resolve().parent
except:
    percorso_cartella = Path.cwd()

n_transazioni = 1_000_000
rng = np.random.default_rng(seed = 123)

# Inizializziamo a None il writer di PyArrow. 
# Verrà attivato solo al primo blocco utile per impostare lo schema del file Parquet.
writer = None

# Generazione dei dati casuali per popolare il dataset
ids = [f"Tran_{i:010d}" for i in range(1, n_transazioni + 1)]
prodotti = ["Smartphone", "Smartwatch", "Laptop", "Tablet", "Pc Desktop", "Cuffie bluetooth", "Cuffie"]
regioni_italiane = [
    "Abruzzo", "Basilicata", "Calabria", "Campania", "Emilia-Romagna",
    "Friuli-Venezia Giulia", "Lazio", "Liguria", "Lombardia", "Marche",
    "Molise", "Piemonte", "Puglia", "Sardegna", "Sicilia", "Toscana",
    "Trentino-Alto Adige", "Umbria", "Valle d'Aosta", "Veneto"
]
prezzi = rng.uniform(10.0, 1000.0, n_transazioni)
quantita = rng.integers(1, 10, n_transazioni)

# Creazione del DataFrame Pandas iniziale
store_df2 = pd.DataFrame({
    "ID": ids,
    "Prodotto": rng.choice(prodotti, n_transazioni),
    "Regione": rng.choice(regioni_italiane, n_transazioni),
    "Prezzo": prezzi.round(2),
    "Valuta": "EUR",
    "Quantita": quantita
})

# Salvataggio iniziale del DataFrame in formato CSV su disco
store_df2.to_csv(percorso_cartella/'store_2.csv', index=False)


# ==========================================
# 2. ELABORAZIONE A CHUNK E SCRITTURA INCREMENTALE
# ==========================================

# Creazione dell'iteratore per leggere il CSV a blocchi di 100.000 righe per volta
chunk_iter = pd.read_csv(percorso_cartella/'store_2.csv', chunksize=100_000)

try:
    for chunk in chunk_iter:
        # Filtriamo il chunk corrente mantenendo solo i prezzi strettamente maggiori di 50
        chunk_filtrato = chunk[chunk["Prezzo"] > 50]

        # Procediamo alla scrittura solo se il chunk filtrato non è vuoto
        if not chunk_filtrato.empty:
            # Convertiamo il DataFrame Pandas filtrato in una Tabella PyArrow nativa
            table = pa.Table.from_pandas(chunk_filtrato)

            # Al primo chunk utile, inizializziamo il ParquetWriter collegandolo al file di output
            if writer is None:
                writer = pq.ParquetWriter(percorso_cartella/'store_2.parquet', table.schema)

            # Scriviamo il blocco corrente in coda (append in-flight) nel file Parquet
            writer.write_table(table)
            
finally:
    # Il blocco finally garantisce la chiusura sicura del file Parquet alla fine del ciclo
    if writer is not None:
        writer.close()
        print("\nProcesso completato! File Parquet salvato con successo.")


# ==========================================
# 3. VERIFICA DELLA CORRETTEZZA DEL FILE FINALE
# ==========================================

# Leggiamo i metadati del file Parquet per contare le righe in modo ultra-veloce (senza caricare i dati in RAM)
parquet_file = pq.ParquetFile(percorso_cartella/'store_2.parquet')
totale_righe_parquet = parquet_file.metadata.num_rows
print(f"Numero totale di righe nel file Parquet: {totale_righe_parquet:,}")

# Confrontiamo il conteggio con il dataset originale filtrato a monte via Pandas
righe_attese = len(store_df2[store_df2["Prezzo"] > 50])
print(f"Numero righe attese (dataframe originale filtrato): {righe_attese:,}")

# Controllo di coerenza finale
if totale_righe_parquet == righe_attese:
    print("\n✅ VERIFICA SUPERATA: Il numero di righe nel file Parquet coincide perfettamente!")
else:
    print("\n❌ ATTENZIONE: C'è una discrepanza nel numero di righe.")

In [18]:
""" 
Esercizio 4
Implementare un semplice streaming simulato: 
leggere righe di un file csv una per una usando un iteratore python 
e calcolare la somma cumulativa della colonna quantita.

"""

import csv
from pathlib import Path

# Percorso del file CSV creato in precedenza 
percorso_csv = percorso_cartella / 'store_2.csv'

# Inizalizziamo la variabile per la somma colulativa
somma_cumulativa = 0

print("\nINIZIO STREAMING DATI\n")

# Apriamo il file e usiamo csv.DictReader come iteratore riga per riga
with open(percorso_csv, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)

    for i, row in enumerate(reader, start=1):
        # estraiamo la quantità della riga corrente ( convertendola in intero)
        quantita_corrente = int(row['Quantita'])

        # aggiorniamo la somma cumulativa
        somma_cumulativa += quantita_corrente

        # stampiamo i primi eventi e poi ogni 250.000 righe 
        if i <= 5 or i % 250_000 == 0:
            print(f"Evento #{i} | ID: {row['ID']} | Q.tà: {quantita_corrente} | Somma Cumulativa: {somma_cumulativa:,}")

print(f"\n--- STREAMING TERMINATO ---")
print(f"Somma cumulativa totale di tutte le quantità: {somma_cumulativa:,}")


INIZIO STREAMING DATI

Evento #1 | ID: Tran_0000000001 | Q.tà: 8 | Somma Cumulativa: 8
Evento #2 | ID: Tran_0000000002 | Q.tà: 6 | Somma Cumulativa: 14
Evento #3 | ID: Tran_0000000003 | Q.tà: 7 | Somma Cumulativa: 21
Evento #4 | ID: Tran_0000000004 | Q.tà: 5 | Somma Cumulativa: 26
Evento #5 | ID: Tran_0000000005 | Q.tà: 8 | Somma Cumulativa: 34
Evento #250000 | ID: Tran_0000250000 | Q.tà: 8 | Somma Cumulativa: 1,251,001
Evento #500000 | ID: Tran_0000500000 | Q.tà: 7 | Somma Cumulativa: 2,500,557
Evento #750000 | ID: Tran_0000750000 | Q.tà: 6 | Somma Cumulativa: 3,752,233
Evento #1000000 | ID: Tran_0001000000 | Q.tà: 6 | Somma Cumulativa: 5,001,375

--- STREAMING TERMINATO ---
Somma cumulativa totale di tutte le quantità: 5,001,375
